# `portopt` — Quick Tour

Este notebook reproduz o espírito dos notebooks do Prof. Chagas, mas usando o `portopt` para abstrair o boilerplate. Mesma matemática, código uma ordem de magnitude menor.

Estrutura:
1. Carregar dados (yfinance ou sintético)
2. **Menu de modelos** — todos os modelos do curso, mesma interface
3. Comparativo lado-a-lado (`compare`)
4. Backtest com diferentes modelos de custo (`BacktestEngine` + `CostModel`)
5. Visualizações

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import portopt as po
from portopt.costs import FlatCost, B3RealisticCost, OffshoreCost, CompositeCost

%matplotlib inline
pd.options.display.float_format = '{:.4f}'.format

## 1. Dados (escolha uma fonte)

In [ ]:
# Online — yfinance:
tickers = ['SPY', 'QQQ', 'IWM', 'TLT', 'GLD', 'EFA']
prices = po.data.load_prices(tickers, start='2018-01-01', end='2024-12-31')
log_rets = po.returns.to_log_returns(prices)
print(f'{len(prices)} dias x {len(tickers)} ativos')
prices.tail()

## 2. Menu de modelos

Todos os modelos compartilham a mesma interface: `model.fit(returns, constraints)`.

In [ ]:
po.models.list_models()

In [ ]:
constraints = po.ConstraintSet(bounds=(0.0, 0.4), sum_to=1.0)

# Tier 0 — Naive
ew_result = po.models.EqualWeight().fit(log_rets, constraints)

# Tier 1 — Markowitz
mv_result = po.models.Markowitz().fit(log_rets, constraints)

# Tier 2 — Mean-Absolute Deviation (linprog HiGHS, conforme Chagas §3.2)
mad_result = po.models.MAD().fit(log_rets, constraints)

# Tier 3 — Hierarchical Risk Parity (Prado 2016)
hrp_result = po.models.HierarchicalRiskParity().fit(log_rets, constraints)

# Tier 3 — Risk Budgeting por grupo
rb_constraints = po.ConstraintSet(
    bounds=(0.0, 0.4),
    asset_groups={'EQ': ['SPY', 'QQQ', 'IWM', 'EFA'], 'RF': ['TLT'], 'GOLD': ['GLD']},
    group_risk_budgets={'EQ': 0.50, 'RF': 0.30, 'GOLD': 0.20},
)
rb_result = po.models.RiskBudget(approach='2').fit(log_rets, rb_constraints)

for name, r in [('EW', ew_result), ('MV', mv_result), ('MAD', mad_result),
                ('HRP', hrp_result), ('RB', rb_result)]:
    print(f'{name:5s}: risk({r.risk_measure})={r.risk:.5f}, top weights:')
    print(r.weights.sort_values(ascending=False).head(3))
    print()

## 3. Comparativo unificado

In [ ]:
comparison = po.compare(
    models=['ew', 'markowitz', 'mad', 'erc', 'hrp', 'cvar'],
    prices=prices,
    constraints=po.ConstraintSet(bounds=(0.0, 0.4)),
)
comparison.weights_table().style.format('{:.3f}').background_gradient(cmap='viridis')

## 4. Backtest com custos plugáveis

Os custos são componentes independentes do modelo (e do engine). Você pode comparar como o mesmo Markowitz se comporta sob 3 stacks de custo diferentes:

In [ ]:
cost_models = {
    'Zero cost (teórico)': po.costs.ZeroCost(),
    'Flat 15 bps (Chagas)': FlatCost(rate=0.0015),
    'B3 realista': B3RealisticCost(),
    'Offshore (US+FX)': OffshoreCost(),
    'B3 + IR': CompositeCost([B3RealisticCost(), po.costs.TaxAwareCost()]),
}

wealth = {}
for label, cost in cost_models.items():
    cfg = po.BacktestConfig(
        training_window=252,
        rebalance='monthly',
        transaction_costs=cost,
        progress=False,
    )
    bt = po.BacktestEngine(cfg).run(prices, po.models.Markowitz(), constraints)
    wealth[label] = bt.cumulative_wealth
    print(f'{label:24s} → Sharpe {bt.metrics["sharpe"]:.3f}, '
          f'MDD {bt.metrics["max_drawdown"]:.2%}, '
          f'custos pagos {bt.costs_paid.sum():.4%}')

po.viz.plot_cumulative_wealth(wealth, title='Markowitz under different cost stacks')

## 5. Comparativo de modelos com backtest

In [ ]:
full_comparison = po.compare(
    models=['ew', 'iv', 'markowitz', 'mad', 'hrp', 'cvar'],
    prices=prices,
    constraints=po.ConstraintSet(bounds=(0.0, 0.4)),
    with_backtest=True,
    backtest_config=po.BacktestConfig(
        training_window=252,
        rebalance='monthly',
        transaction_costs=FlatCost(rate=0.0015),
        progress=False,
    ),
)
full_comparison.metrics_table().style.format('{:.4f}').background_gradient(cmap='RdYlGn')

In [ ]:
po.viz.plot_cumulative_wealth(full_comparison.cumulative_wealth(),
                              title='Backtest comparison')